<a href="https://colab.research.google.com/github/Pujitha-pitta/Flyrank-ML-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


This project is a binary classification task where the goal is to predict whether a content page needs to be refreshed.

I selected Logistic Regression because it is simple, interpretable, and performs well for binary classification problems. It provides a strong baseline machine learning model and allows feature importance to be interpreted through model coefficients.

The model will be compared with the Week 4 rule-based baseline using the same dataset, train-test split, and evaluation metrics.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



The dataset is split into training and testing sets using an 80/20 split.

A grouped split based on client_id is used so that pages from the same client do not appear in both the training and testing datasets. This provides a more realistic evaluation because the model is tested on unseen clients.

The same split is used for both the Week 4 baseline and the Logistic Regression model to ensure a fair comparison.

In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/Pujitha-pitta/Flyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
import numpy as np

df["needs_refresh"] = (
    df["trend_direction"] == "down"
).astype(int)

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "sessions_90d",
    "search_volume"
]

X = df[features].replace([np.inf, -np.inf], np.nan)
y = df["needs_refresh"]
groups = df["client_id"]

print("Target distribution:")
print(y.value_counts())

Target distribution:
needs_refresh
1    16262
0    13738
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Testing clients:", groups.iloc[test_idx].nunique())

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



I trained a Logistic Regression model because it is suitable for binary classification and remains interpretable.

The model is compared with a simple rule-based baseline using the same test data and the same F1 metric. The goal is not to reward complexity, but to check whether the learned model provides a measurable improvement.

In [8]:
baseline_pred = (
    (
        (X_test["ctr"] < X_train["ctr"].median()) &
        (X_test["impressions_90d"] > X_train["impressions_90d"].median())
    )
    |
    (
        (X_test["days_since_last_update"] >
         X_train["days_since_last_update"].median()) &
        (X_test["engagement_rate"] <
         X_train["engagement_rate"].median())
    )
).astype(int)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_pred = model.predict(X_test)
model_prob = model.predict_proba(X_test)[:, 1]

In [10]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate(y_true, y_pred, y_prob=None):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_true,
            y_prob if y_prob is not None else y_pred
        )
    }

baseline_results = evaluate(
    y_test,
    baseline_pred
)

model_results = evaluate(
    y_test,
    model_pred,
    model_prob
)

comparison = pd.DataFrame(
    [baseline_results, model_results],
    index=[
        "Week-4 Rule Baseline",
        "Logistic Regression"
    ]
)

comparison.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC
Week-4 Rule Baseline,0.515,0.628,0.124,0.206,0.524
Logistic Regression,0.534,0.548,0.511,0.529,0.541


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



The Logistic Regression model improved overall performance compared with the Week 4 rule-based baseline. The largest improvement was observed in the F1 Score, which increased from 0.206 to 0.529, indicating a better balance between precision and recall.

The model also achieved a much higher recall (0.511 compared with 0.124), meaning it identified more content pages that may require refreshing. This is useful because missing pages that need updates can reduce search performance.

Some prediction errors still occur for pages with mixed signals, such as high impressions but stable engagement, or older pages that continue to perform well. These cases are difficult to classify because the available features provide conflicting evidence.

The model relies on behavioural features such as click-through rate, impressions, average position, engagement rate, and content age. These features provide useful signals for identifying pages that may benefit from refreshing.

The model should be used as a decision-support tool to help prioritize content reviews rather than making automatic publishing decisions.

In [11]:
coefficients = model.named_steps["classifier"].coef_[0]

importance = pd.DataFrame({
    "Feature": features,
    "Coefficient": coefficients
})

importance = importance.sort_values(
    by="Coefficient",
    ascending=False
)

importance

,Feature,Coefficient
6,days_since_last_update,0.249846
0,impressions_90d,0.043149
8,search_volume,0.006467
3,avg_position,-0.005480
4,engagement_rate,-0.010957
7,sessions_90d,-0.057088
1,clicks_90d,-0.137790
2,ctr,-0.193497
5,content_age_days,-0.396424


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.